<a href="https://colab.research.google.com/github/tecepeipe/ollama-colab-runner/blob/main/ollama_colab_runner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Ollama Colab Runner**
# <img src='https://ollama.com/public/ollama.png' alt="Ollama"/>
When running this, ideally, select an instance with GPU:<br>
T4 for free ones, A100/L4 for paid subscribers<br><br>
Run each of the 3 cells, before running your prompt.<br>
If you interrupt execution, start the server again

In [ ]:
# @title Install components
!apt-get install -y pciutils lshw
!apt-get update
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install ollama

!echo 'debconf debconf/frontend select Noninteractive' | sudo debconf-set-selections
!sudo apt-get update && sudo apt-get install -y cuda-drivers

import os
# Set LD_LIBRARY_PATH so the system NVIDIA library
os.environ.update({'LD_LIBRARY_PATH': '/usr/lib64-nvidia'})

In [ ]:
# @title Start server and API endpoint
import subprocess
import os

env = os.environ.copy()
env["OLLAMA_HOST"] = "0.0.0.0"
env["OLLAMA_ORIGINS"] = "*"
process = subprocess.Popen(['ollama', 'serve'], env=env)

In [ ]:
# @title Select your model
model = "gemma4:e4b" # @param ["gemma4:e4b","deepseek-r1:1.5b","deepseek-r1:7b","deepseek-r1:14b","deepseek-r1:32b","deepseek-r1:70b","deepseek-coder:1.3b","deepseek-coder:6.7b","deepseek-coder:33b","gemma3:12b","gemma3:27b","llama3.3:70b","mistral:7b","phi4:14b","qwen2.5:7b","qwen2.5:14b","qwen2.5:32b","qwen2.5-coder:7b","qwen2.5-coder:14b","qwen2.5-coder:32b"]
!ollama pull {model}

In [ ]:
# @title Interacting with the model (streaming)
Prompt = ""  # @param {"type":"string"}

from IPython.display import display, Markdown
import ollama

# Start streaming response
stream = ollama.chat(
    model=model,
    messages=[
        {
            'role': 'user',
            'content': Prompt,
        },
    ],
    stream=True,
)

# Collect and display streamed chunks
full_response = ""

for chunk in stream:
    content = chunk['message']['content']
    full_response += content
    print(content, end="", flush=True)

# Optional Markdown rendering after completion
display(Markdown(full_response))

In [ ]:
# @title Exposing Ollama REST API to use it as local LLM or VS Code
!npm install -g localtunnel # Cloudflare free Tunnels
# Start a background tunnel pointing to Ollama's default port
!lt --port 11434

In [ ]:
# @title If changing models, cancel the tunnel and execute this to kill Ollama
!pkill ollama